# CAPE-TTA × OpenVLA × LIBERO — Free-Colab Oracle Pilot

This notebook runs a **real OpenVLA checkpoint** on LIBERO and evaluates K counterfactual first-action candidates with CAPE-TTA's validated fresh-environment prefix-replay oracle.

**Pilot only:** the run uses 8-bit OpenVLA + FP16 on a free T4-class GPU to fit memory. Official LIBERO sparse success and CAPE's simulation-only soft task potential are logged separately. Do not report this pilot as the final paper benchmark.


In [6]:
import subprocess, shutil, sys, os
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU, then rerun.')
subprocess.run(['nvidia-smi'], check=True)
print('Host Python:', sys.version)


RuntimeError: No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU, then rerun.

## 1. Build an isolated Python 3.10 environment
The notebook uses micromamba so Colab host-Python changes do not alter the OpenVLA environment.


In [ ]:
import os, subprocess, textwrap
def sh(cmd):
    print('\n$', cmd)
    subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')

sh('apt-get update -qq')
sh('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq curl git ffmpeg libosmesa6-dev libgl1-mesa-glx libglfw3 build-essential')
if not os.path.exists('/content/micromamba'):
    sh("cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba && mv bin/micromamba /content/micromamba")
if not os.path.exists('/content/cape-env/bin/python'):
    sh('/content/micromamba create -y -p /content/cape-env -c conda-forge python=3.10 pip')
PY='/content/cape-env/bin/python'
PIP=f'{PY} -m pip'
sh(f'{PIP} install --upgrade pip')
sh(f'{PIP} install --index-url https://download.pytorch.org/whl/cu121 torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0')
sh(f'''{PIP} install \
  numpy==1.26.4 scipy==1.11.4 matplotlib==3.7.5 \
  transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate==0.29.3 \
  bitsandbytes==0.43.1 peft==0.11.1 sentencepiece==0.1.99 einops rich protobuf \
  tensorflow==2.15.0 pillow==10.3.0 safetensors huggingface_hub \
  'imageio[ffmpeg]' robosuite==1.4.1 bddl easydict cloudpickle gym==0.25.2''')
print('Environment ready:', PY)


## 2. Clone CAPE-TTA and LIBERO, then configure LIBERO


In [ ]:
import os, subprocess
def sh(cmd):
    print('\n$', cmd)
    subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')
PY='/content/cape-env/bin/python'
PIP=f'{PY} -m pip'
sh('rm -rf /content/CAPE-TTA /content/LIBERO')
sh('git clone --depth 1 https://github.com/weichezhang/CAPE-TTA.git /content/CAPE-TTA')
sh('git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO')
sh(f'{PIP} install -e /content/CAPE-TTA --no-deps')
sh(f'{PIP} install -e /content/LIBERO --no-deps')
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
cfg='''benchmark_root: /content/LIBERO/libero/libero\nbddl_files: /content/LIBERO/libero/libero/bddl_files\ninit_states: /content/LIBERO/libero/libero/init_files\ndatasets: /content/LIBERO/libero/datasets\nassets: /content/LIBERO/libero/libero/assets\n'''
open(os.path.expanduser('~/.libero/config.yaml'),'w').write(cfg)
commit=subprocess.check_output(['git','-C','/content/CAPE-TTA','rev-parse','HEAD'], text=True).strip()
print('CAPE-TTA commit:', commit)


## 3. GPU/runtime preflight


In [ ]:
import subprocess, textwrap
PY='/content/cape-env/bin/python'
code=r'''import torch, transformers, tokenizers, timm, bitsandbytes as bnb, numpy as np
print('torch=', torch.__version__, 'cuda=', torch.version.cuda)
print('gpu=', torch.cuda.get_device_name(0), 'capability=', torch.cuda.get_device_capability(0))
print('transformers=', transformers.__version__, 'tokenizers=', tokenizers.__version__, 'timm=', timm.__version__, 'numpy=', np.__version__, 'bnb=', bnb.__version__)
assert torch.cuda.is_available()
'''
subprocess.run([PY,'-c',code], check=True)


## 4. Run the real OpenVLA → LIBERO → Oracle-CAPE pilot
Configuration: LIBERO-Spatial task 0, init state 0, K=4 candidates, horizon=12. The base action is greedy OpenVLA; alternatives are sampled only from legal OpenVLA action-token IDs.


In [7]:
import os, subprocess, json, pathlib
PY='/content/cape-env/bin/python'
commit=subprocess.check_output(['git','-C','/content/CAPE-TTA','rev-parse','HEAD'], text=True).strip()
env=os.environ.copy()
env.update({
    'MUJOCO_GL':'osmesa',
    'PYOPENGL_PLATFORM':'osmesa',
    'LIBERO_REPO_ROOT':'/content/LIBERO',
    'CAPE_COMMIT':commit,
    'TOKENIZERS_PARALLELISM':'false',
    'TF_CPP_MIN_LOG_LEVEL':'2',
    'BITSANDBYTES_NOWELCOME':'1',
    'HF_HOME':'/content/hf_cache',
})
out='/content/CAPE-TTA/results/openvla_oracle_pilot_colab.json'
cmd=[PY,'/content/CAPE-TTA/scripts/run_openvla_oracle_pilot.py',
     '--checkpoint','openvla/openvla-7b-finetuned-libero-spatial',
     '--suite','libero_spatial','--task-id','0','--init-state-id','0',
     '--candidates','4','--horizon','12','--settle-steps','10',
     '--temperature','0.7','--top-p','0.95','--seed','7',
     '--no-flash-attention','--load-in-8bit','--input-dtype','fp16',
     '--output',out]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, env=env, check=True)
result=json.load(open(out))
print('\n=== PILOT SUMMARY ===')
print('GPU:', result['runtime']['gpu'])
print('wrapper parity max abs:', result['wrapper_parity_max_abs'])
print('fresh replay:', result['fresh_replay_determinism'])
print('base soft gain:', result['base_soft_gain'])
print('oracle best index:', result['oracle_best_index'])
print('oracle best soft gain:', result['oracle_best_soft_gain'])
print('oracle margin over base:', result['oracle_margin_over_base'])


CalledProcessError: Command '['git', '-C', '/content/CAPE-TTA', 'rev-parse', 'HEAD']' returned non-zero exit status 128.

## 5. Inspect and download the raw JSON


In [ ]:
import json
from google.colab import files
out='/content/CAPE-TTA/results/openvla_oracle_pilot_colab.json'
data=json.load(open(out))
print(json.dumps(data, indent=2)[:12000])
files.download(out)


### What to do with the JSON
Upload `openvla_oracle_pilot_colab.json` back to the CAPE-TTA ChatGPT project. The next stage is to verify candidate diversity and oracle margin, then scale to more init states/tasks and fit the conformal calibration split. Do **not** treat this single pilot as the paper's final benchmark result.
